# 模型评估

In [ ]:
import sys; sys.path.append("..")
import os, torch, numpy as np
from torch.utils.data import DataLoader
from src.config import Config; from src.utils.seed import set_seed
cfg = Config(); set_seed(cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from src.data.dataset import OralFluorescenceDataset
from src.data.transforms import get_val_transforms
tf = get_val_transforms()
ts_ds = OralFluorescenceDataset(cfg.data.data_root, os.path.join(cfg.data.split_dir, cfg.data.test_split), transform=tf, class_names=cfg.data.class_names)
ts_l = DataLoader(ts_ds, cfg.training.batch_size, shuffle=False, num_workers=0)
print(f"Test: {len(ts_ds)}")

In [ ]:
from src.models.backbone import create_model
model = create_model(cfg)
ckpt = "checkpoints/baseline_v1/best_model.pth"
if os.path.exists(ckpt):
    m = torch.load(ckpt, map_location=device)
    model.load_state_dict(m["model_state_dict"])
    model.to(device)
    print(f"Loaded: {m['accuracy']:.2f}%")

In [ ]:
from src.evaluation.metrics import evaluate, compute_metrics
from src.evaluation.visualization import plot_confusion_matrix, plot_roc_curves
y_true, y_pred, y_prob = evaluate(model, ts_l, device)
metrics = compute_metrics(y_true, y_pred, y_prob, cfg.data.num_classes)
print(f"Acc: {metrics['accuracy']:.4f} | F1: {metrics['f1_macro']:.4f} | AUC: {metrics.get('auc_macro', 'N/A')}")
plot_confusion_matrix(metrics['confusion_matrix'], cfg.data.class_names)
plt.show()
plot_roc_curves(y_true, y_prob, cfg.data.class_names)
plt.show()